In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dejintime/moviereviews/testData.tsv/testData.tsv
/kaggle/input/datasets/dejintime/moviereviews/labeledTrainData.tsv/labeledTrainData.tsv


### 1 read data

In [2]:
train_data = pd.read_csv("/kaggle/input/datasets/dejintime/moviereviews/labeledTrainData.tsv", header = 0, delimiter = '\t', quoting=3)
train_data.iloc[0]["review"]

'"With all this stuff going down at the moment with MJ i\'ve started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ\'s feeling towards the press and also the obvious message of drugs are bad m\'kay.<br /><br />Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.<br /><br />The actual feature film bit when it finally

### 2 clean data

In [ ]:
from bs4 import BeautifulSoup
string_exp = BeautifulSoup(train_data.iloc[0]["review"])
print(string_exp.get_text(),"\n")
print(train_data.iloc[0]["review"])

In [ ]:
import re
letters_only = re.sub("[^a-zA-Z]", " ", string_exp.get_text())
letters_only

In [ ]:
lower_ = letters_only.lower()
words = lower_.split()

In [ ]:
import nltk
nltk.download()

In [ ]:
from nltk.corpus import stopwords
print(stopwords.words("english"))

In [ ]:
words = [w for w in words if w not in stopwords.words("english")]
print(words)

#### create a function to clean data and transform to words

In [3]:
from bs4 import BeautifulSoup
import re
from nltk.corpus import stopwords
def CleanData(raw_review):
    review_text = BeautifulSoup(raw_review).get_text() # remove HTML
    review_letters_only = re.sub("[^a-zA-Z]", " ", review_text)
    review_letters_only = review_letters_only.lower()
    words = review_letters_only.split()
    words = [w for w in words if w not in stopwords.words("english")]
    return " ".join(words)

In [4]:
CleanData(train_data.iloc[0]["review"]) # test this function

'stuff going moment mj started listening music watching odd documentary watched wiz watched moonwalker maybe want get certain insight guy thought really cool eighties maybe make mind whether guilty innocent moonwalker part biography part feature film remember going see cinema originally released subtle messages mj feeling towards press also obvious message drugs bad kay visually impressive course michael jackson unless remotely like mj anyway going hate find boring may call mj egotist consenting making movie mj fans would say made fans true really nice actual feature film bit finally starts minutes excluding smooth criminal sequence joe pesci convincing psychopathic powerful drug lord wants mj dead bad beyond mj overheard plans nah joe pesci character ranted wanted people know supplying drugs etc dunno maybe hates mj music lots cool things like mj turning car robot whole speed demon sequence also director must patience saint came filming kiddy bad sequence usually directors hate workin

In [6]:
print("Now let's begin clean all the data....\n")
review_nums = train_data["review"].size
clean_train_data = []
for i in range(review_nums):
    s = CleanData(train_data.iloc[i]["review"])
    if((i+1)%1000 == 0):
        print(f"review:{i+1} / {review_nums}")
    clean_train_data.append(s)

Now let's begin clean all the data....

review:1000 / 25000
review:2000 / 25000
review:3000 / 25000
review:4000 / 25000
review:5000 / 25000
review:6000 / 25000
review:7000 / 25000
review:8000 / 25000
review:9000 / 25000
review:10000 / 25000
review:11000 / 25000
review:12000 / 25000
review:13000 / 25000
review:14000 / 25000
review:15000 / 25000
review:16000 / 25000
review:17000 / 25000
review:18000 / 25000
review:19000 / 25000
review:20000 / 25000
review:21000 / 25000
review:22000 / 25000
review:23000 / 25000
review:24000 / 25000
review:25000 / 25000


### 3 Create the bags of words

In [18]:
from sklearn.feature_extraction.text import CountVectorizer
print("Use sklearn to create the bags of words...\n")
vectorizer = CountVectorizer(analyzer = "word", tokenizer = None, stop_words = None, preprocessor = None, max_features = 5000)
train_data_features = vectorizer.fit_transform(clean_train_data)
train_data_features = train_data_features.toarray()

Create the bags of words...



### 4 Use random forest training features

In [21]:
print("Use random forest to train features...\n")
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(n_estimators = 100)
forest = forest.fit(train_data_features, train_data["sentiment"])

Use random forest to train features...

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [26]:
test_data = pd.read_csv("/kaggle/input/datasets/dejintime/moviereviews/labeledTrainData.tsv/labeledTrainData.tsv", 
                        header = 0, 
                        delimiter = '\t', 
                        quoting = 3)
test_review_nums = len(test_data["review"])
clean_test_data = []
print("Begin to clean test data...\n")
for i in range(test_review_nums):
    s = CleanData(test_data.iloc[i]["review"])
    if((i+1)%1000 == 0):
        print(f"review:{i+1} / {test_review_nums}")
    clean_test_data.append(s)

Begin to clean test data...

review:1000 / 25000
review:2000 / 25000
review:3000 / 25000
review:4000 / 25000
review:5000 / 25000
review:6000 / 25000
review:7000 / 25000
review:8000 / 25000
review:9000 / 25000
review:10000 / 25000
review:11000 / 25000
review:12000 / 25000
review:13000 / 25000
review:14000 / 25000
review:15000 / 25000
review:16000 / 25000
review:17000 / 25000
review:18000 / 25000
review:19000 / 25000
review:20000 / 25000
review:21000 / 25000
review:22000 / 25000
review:23000 / 25000
review:24000 / 25000
review:25000 / 25000


In [33]:
test_data_features = vectorizer.transform(clean_test_data)
test_data_featutes = test_data_features.toarray()
result = forest.predict(test_data_features)
output = pd.DataFrame(data = {"id":test_data["id"], "sentiment":result})
import os 
os.makedirs("/kaggle/working/output", exist_ok = True)
output.to_csv("/kaggle/working/output/Bag_of_Words_model.csv", index = False, quoting = 3)